In [33]:
import os
from dotenv import load_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI
env_path = "../.env"

load_dotenv(dotenv_path=env_path, override=True)
API_KEY = os.getenv("GOOGLE_API_KEY")

if API_KEY:
    chat = ChatGoogleGenerativeAI(model="gemini-2.5-flash-lite")
    print("API Key loaded successfully.")
else:
    print("API Key not found. Please set the GOOGLE_API_KEY in the .env file.")
    


API Key loaded successfully.


In [34]:
from langchain_core.prompts import ChatPromptTemplate, SystemMessagePromptTemplate, AIMessagePromptTemplate
# Function to generate random historical questions to users

def generate_historical_question():
    sys_msg = "You're an expert QA bot that generates random historical questions for users to answer and the questions should always be of DATE related. Make sure to strictly ask the questions without providing any answers or additional information and question should always be concise and to the point. Always ask one question at a time."

    sys_msg_template = SystemMessagePromptTemplate.from_template(sys_msg)
    # ai_msg = "What year did the Roman Empire Officially fall?"
    # ai_msg_template = AIMessagePromptTemplate.from_template(ai_msg)
    
    # ai_msg_2 = "Jesus Christ was born in which year?"
    # ai_msg_template_2 = AIMessagePromptTemplate.from_template(ai_msg_2)
    prompt = ChatPromptTemplate.from_messages([sys_msg_template])
    formatted_prompt = prompt.format_prompt().to_messages()
    
    response = chat.invoke(formatted_prompt[0].content)
    
    print(response.content)
    
generate_historical_question()

In what year did the Titanic sink?


In [46]:
from pydantic import BaseModel, Field
from datetime import datetime as Datetime
from langchain_core.output_parsers import PydanticOutputParser

class date_time_parser(BaseModel):
    status: str = Field(description="Indicates whether the answer is 'Correct' or 'Incorrect'.")
    date: Datetime = Field(description="The date parsed from the user's answer.")
    

output_parser = PydanticOutputParser(pydantic_object=date_time_parser)

def check_qa_bot_response(question: str, answer: str):
    sys_msg = "You're an expert QA bot that specializes in checking the correctness of the historical question generated by another bot. Your task is to evaluate whether the answer to the question is correct or not. If the answer meets this criteria, respond with 'Correct'. If not, respond with 'Incorrect' and provide the correct version of the question. If Incorrect, always provide a concise DATE only, no extra information." \
    "Here is the question: {question} and the answer provided is: {answer}. Determine if the answer is correct or not.\n{format_instructions}"

    sys_msg_template = SystemMessagePromptTemplate.from_template(sys_msg)
    prompt = ChatPromptTemplate.from_messages([sys_msg_template])
    formatted_prompt = prompt.format_prompt(question=question, 
                                            answer=answer, 
                                            format_instructions=output_parser.get_format_instructions()).to_messages()
    response = chat.invoke(formatted_prompt[0].content)
    
    output = output_parser.parse(response.content)
    print(output)

check_qa_bot_response("Kargil war end on?", "26th July, 1999") 

status='Correct' date=datetime.datetime(1999, 7, 26, 0, 0, tzinfo=TzInfo(0))
